In [1]:
import polars as pl, duckdb, os
from datetime import datetime

RAW = "../data/processed/data.parquet"
CLEAN = "../data/processed/data_clean.parquet"
lf = pl.scan_parquet(RAW)
n_raw = lf.select(pl.len()).collect().item()
print(f"Baris mentah: {n_raw:,}")
duckdb.sql(f"SUMMARIZE SELECT * FROM '{RAW}'").show(max_width=250)

Baris mentah: 32,735


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬──────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬───────────────┬────────┬────────┬────────┬────────┬────────┬───────┬─────────────────┐
│ column_name │ column_type │                               min                                │                               max                                │ approx_unique │  avg   │  std   │  q25   │  q50   │  q75   │ count │ null_percentage │
│   varchar   │   varchar   │                             varchar                              │                             varchar                              │     int64     │ "null" │ "null" │ "null" │ "null" │ "null" │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼──────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────┼───────────────┼────────┼────────┼────────┼────────┼────────┼───────┼───────────────

In [2]:
def base(c):
    e = pl.col(c).str.strip_chars()
    return pl.when(e == "").then(None).otherwise(e)

sample = lf.head(200_000).collect()
num_cols, date_cols = [], []
for c in sample.columns:
    s = sample.select(base(c).alias(c))[c].drop_nulls()
    if s.len() == 0:
        continue
    if s.cast(pl.Float64, strict=False).drop_nulls().len() / s.len() > 0.95:
        num_cols.append(c); continue
    try:
        if s.str.to_datetime(strict=False).drop_nulls().len() / s.len() > 0.95:
            date_cols.append(c)
    except Exception:
        pass

print("Numerik :", num_cols)
print("Tanggal :", date_cols)

Numerik : ['id']
Tanggal : ['date', 'created_at', 'updated_at']


In [3]:
names = lf.collect_schema().names()
exprs = []
for c in names:
    e = base(c)
    if c in num_cols:
        e = e.cast(pl.Float64, strict=False)
    elif c in date_cols:
        e = (e.str.to_datetime(time_zone="UTC", strict=False)
              .dt.replace_time_zone(None))
    exprs.append(e.alias(c))

step0 = lf.with_columns(exprs)
log = {"mentah": n_raw}

# 1) baris kosong semua
step1 = step0.filter(~pl.all_horizontal(pl.all().is_null()))
# 2) duplikat penuh
step2 = step1.unique(keep="first")
# 3) validasi tanggal: tidak boleh di masa depan / sebelum 1990
step3 = step2
for c in date_cols:
    step3 = step3.filter(
        pl.col(c).is_null() | ((pl.col(c) >= datetime(1990,1,1)) & (pl.col(c) <= datetime.now())))
# 4) validasi numerik: buang nilai tak hingga
for c in num_cols:
    step3 = step3.filter(pl.col(c).is_null() | pl.col(c).is_finite())

for name, q in [("setelah buang baris kosong", step1), ("setelah dedup", step2), ("setelah validasi", step3)]:
    log[name] = q.select(pl.len()).collect().item()
for k, v in log.items():
    print(f"{k:30s}: {v:,}")

mentah                        : 32,735
setelah buang baris kosong    : 32,735
setelah dedup                 : 32,735
setelah validasi              : 32,735


In [4]:
nulls = step3.null_count().collect()
total = log["setelah validasi"]
miss = (pl.DataFrame({"kolom": nulls.columns, "null": nulls.row(0)})
        .with_columns((pl.col("null")/total*100).round(2).alias("persen_null"))
        .sort("persen_null", descending=True))
print(miss)
drop_cols = miss.filter(pl.col("persen_null") > 70)["kolom"].to_list()
print("Kolom dibuang (>70% null):", drop_cols)
step4 = step3.drop(drop_cols)
num_cols = [c for c in num_cols if c not in drop_cols]

shape: (11, 3)
┌────────────┬──────┬─────────────┐
│ kolom      ┆ null ┆ persen_null │
│ ---        ┆ ---  ┆ ---         │
│ str        ┆ i64  ┆ f64         │
╞════════════╪══════╪═════════════╡
│ content    ┆ 441  ┆ 1.35        │
│ image      ┆ 160  ┆ 0.49        │
│ summary    ┆ 10   ┆ 0.03        │
│ id         ┆ 0    ┆ 0.0         │
│ source     ┆ 0    ┆ 0.0         │
│ …          ┆ …    ┆ …           │
│ url        ┆ 0    ┆ 0.0         │
│ date       ┆ 0    ┆ 0.0         │
│ embedding  ┆ 0    ┆ 0.0         │
│ created_at ┆ 0    ┆ 0.0         │
│ updated_at ┆ 0    ┆ 0.0         │
└────────────┴──────┴─────────────┘
Kolom dibuang (>70% null): []


In [5]:
bounds = {}
for c in num_cols:
    q = step4.select(pl.col(c).quantile(0.25).alias("q1"), pl.col(c).quantile(0.75).alias("q3")).collect()
    q1, q3 = q["q1"][0], q["q3"][0]
    if q1 is None or q3 is None: continue
    iqr = q3 - q1
    bounds[c] = (q1 - 1.5*iqr, q3 + 1.5*iqr)

flags = [((pl.col(c) < lo) | (pl.col(c) > hi)).fill_null(False).alias(f"out_{c}")
         for c, (lo, hi) in bounds.items()]
step5 = step4.with_columns(flags) if flags else step4

if flags:
    cnt = step5.select([pl.col(f"out_{c}").sum().alias(c) for c in bounds]).collect()
    print("Jumlah outlier per kolom:"); print(cnt)

Jumlah outlier per kolom:
shape: (1, 1)
┌─────┐
│ id  │
│ --- │
│ u32 │
╞═════╡
│ 0   │
└─────┘


In [6]:
step5.collect(engine="streaming").write_parquet(CLEAN)
n_clean = pl.scan_parquet(CLEAN).select(pl.len()).collect().item()
print(f"Mentah {n_raw:,} -> Bersih {n_clean:,} (dibuang {(n_raw-n_clean)/n_raw:.1%})")
duckdb.sql(f"SUMMARIZE SELECT * FROM '{CLEAN}'").show(max_width=250)

Mentah 32,735 -> Bersih 32,735 (dibuang 0.0%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬─────────────────────────┬─────────────────────────┬───────────────┬──────────────────────────┬────────────────────┬──────────────────────────┬─────────────────────────┬─────────────────────────┬───────┬─────────────────┐
│ column_name │ column_type │           min           │           max           │ approx_unique │           avg            │        std         │           q25            │           q50           │           q75           │ count │ null_percentage │
│   varchar   │   varchar   │         varchar         │         varchar         │     int64     │         varchar          │      varchar       │         varchar          │         varchar         │         varchar         │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼─────────────────────────┼─────────────────────────┼───────────────┼──────────────────────────┼────────────────────┼──────────────────────────┼─────────────────────────┼─────────────────────────┼───────┼───────────────